In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import Binarizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.pipeline import Pipeline

# 1. Load sample dataset
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

print(f"Dataset shape: {X.shape}")
print(f"First 3 rows of raw continuous data:\n{X[['mean radius', 'mean texture', 'mean perimeter']].head(3)}\n")

# 2. Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ---------------------------------------------------------
# Example A: Standalone Binarization
# ---------------------------------------------------------
# Set a custom threshold (e.g., threshold=15.0 means values > 15 become 1, <= 15 become 0)
binarizer = Binarizer(threshold=15.0)

# Transforming a subset of features
sample_feature = X_train[['mean radius']]
binarized_feature = binarizer.fit_transform(sample_feature)

demo_df = pd.DataFrame({
    'Original Value': sample_feature['mean radius'].values[:5],
    'Binarized (> 15.0)': binarized_feature.flatten()[:5]
})
print("Standalone Binarization Comparison:")
print(demo_df)
print("-" * 50)

# ---------------------------------------------------------
# Example B: End-to-End Pipeline with ML Classifier
# ---------------------------------------------------------
# Binarize all features at threshold=10.0 and feed directly into Logistic Regression
pipeline = Pipeline([
    ('binarizer', Binarizer(threshold=10.0)),
    ('classifier', LogisticRegression(solver='liblinear', random_state=42))
])

# Fit on training data
pipeline.fit(X_train, y_train)

# Evaluate on test set
y_pred = pipeline.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy * 100:.2f}%\n")
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=data.target_names))

Dataset shape: (569, 30)
First 3 rows of raw continuous data:
   mean radius  mean texture  mean perimeter
0        17.99         10.38           122.8
1        20.57         17.77           132.9
2        19.69         21.25           130.0

Standalone Binarization Comparison:
   Original Value  Binarized (> 15.0)
0           10.32                 0.0
1           20.18                 1.0
2           10.66                 0.0
3           13.56                 0.0
4           11.37                 0.0
--------------------------------------------------
Test Accuracy: 64.04%

Classification Report:
              precision    recall  f1-score   support

   malignant       1.00      0.02      0.05        42
      benign       0.64      1.00      0.78        72

    accuracy                           0.64       114
   macro avg       0.82      0.51      0.41       114
weighted avg       0.77      0.64      0.51       114

